# Pulse-level access: calibrate a custom X gate

This notebook defines its own pulse shape for the single-qubit `prx` gate, calibrates its
amplitude with Rabi sweeps, and compares the result with the machine's native gate. Everything
runs through LEXIS / HEAppE / QScheduler with `qheappe_client.QPulla`, an IQM pulse-level client
(`iqm.iqm_client.IQMClient`), so the IQM compiler and job API are used as IQM documents them.

**Reference.** The experiment follows the calibration workflow of Filippos Dakis,
[Experimental_results_IQM](https://github.com/Filippos-Dakis/Experimental_results_IQM)
(`X_calibration_exp.ipynb`): custom PRX waveforms, a Rabi amplitude calibration in rounds,
and an $X^2 = I$ check against the native gate. That repository runs on IQM Resonance with
iqm-pulla 13 and optimized, leakage-suppressed pulses. This notebook is an independent
implementation for `qaas` (iqm-client 36, iqm-pulla 15) with a simple Hann pulse instead.

Steps:

1. Connect and pick a qubit.
2. Read the qubit's current calibration (native gate, drive frequency, coherence times).
3. Define a Hann pulse with an optional DRAG component and register it as a `prx` implementation.
4. Calibrate its amplitude: three Rabi sweeps, each narrower than the last.
5. Check $X$ (expect $P(1) \approx 1$) and $X^2$ (expect $P(1) \approx 0$) against the native gate.

**Requirements:** `qaas[iqm_qiskit]`, QScheduler v0.2.2+ with pulse access enabled for the
machine, HEAppE 6.7.0+. Every sweep and check is one pulse-level job and consumes allocation
(7 jobs in total). A mock backend returns synthetic counts, so the fitted values are only
meaningful on real hardware.

In [ ]:
%pip install matplotlib
%pip install 'qaas[iqm_qiskit]==1.0.0' --extra-index-url https://opencode.it4i.eu/api/v4/projects/107/packages/pypi/simple

## Configuration

In [ ]:
import json
import os
from dataclasses import dataclass
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
from exa.common.data.parameter import Sweep
from exa.common.errors.iqm_error import NotFoundError
from iqm.models.playlist.waveforms import Waveform
from iqm.pulse.gates.prx import PRX_CustomWaveforms
from iqm.station_control.interface.models.jobs_new import JobStatus
from qiskit import QuantumCircuit

from qheappe_client import QProvider
from qheappe_client.utils import QException

os.environ.setdefault("QPROVIDER_LOGLEVEL", "INFO")

LEXIS_PROJECT = "EU-xx-xxx"  # LEXIS project short name
RESOURCE = "VLQ-EU"  # LEXIS quantum resource
BACKEND = "VLQ"  # backend (cluster) name; None for single-backend resources
QUBIT = None  # e.g. "QB5"; None picks the first qubit of the chip

IMPL = "hann_drag"  # name of the new prx implementation
DRAG_RATIO = 0.0  # amplitude_q / amplitude_i; 0.0 = plain Hann pulse
JOB_TIMEOUT = 900  # seconds to wait for each job

## Connect

`get_pulla()` returns a `QPulla` for the backend. It needs a LEXIS login, like `get_backend()`;
pass `token=...` to `QProvider` to use an existing token.

In [ ]:
provider = QProvider(LEXIS_PROJECT)
pulla = provider.get_pulla(RESOURCE, BACKEND)

qubits = pulla.get_chip_topology().qubits_sorted
QUBIT = QUBIT or qubits[0]
print(
    pulla, "| chip", pulla.get_chip_label(), "|", len(qubits), "qubits | using", QUBIT
)

## Current calibration of the qubit

The Qiskit compiler maps Qiskit qubit 0 onto `QUBIT` when compiling with `components=[QUBIT]`.
The native `prx` implementation gives the pulse duration and a starting amplitude.

In [ ]:
compiler = pulla.get_qiskit_compiler()

settings = compiler.get_settings(circuits=[QuantumCircuit(1, 1)])
NATIVE_IMPL = settings.gate_definitions.prx.default_implementation.value
native_gate = settings.gates.prx[NATIVE_IMPL][QUBIT]
DURATION = float(native_gate.duration.value)  # seconds; reuse the native pulse length
NATIVE_AMPLITUDE = float(native_gate.amplitude_i.value)

info = {
    "native prx implementation": NATIVE_IMPL,
    "pulse duration (ns)": DURATION * 1e9,
    "native amplitude_i": NATIVE_AMPLITUDE,
    "drive frequency (GHz)": settings.controllers[QUBIT].drive.frequency.value / 1e9,
}
try:
    metrics = pulla.get_calibration_quality_metrics()
    t1, t2 = metrics.get_coherence_times([QUBIT])
    if t1.get(QUBIT):
        info["T1 (us)"] = t1[QUBIT] * 1e6
    if t2.get(QUBIT):
        info["T2 (us)"] = t2[QUBIT] * 1e6
    fidelity = metrics.get_gate_fidelity("prx", NATIVE_IMPL, (QUBIT,))
    if fidelity is not None:
        info["native prx fidelity"] = fidelity
except (
    NotFoundError,
    QException,
    KeyError,
    ValueError,
) as e:  # optional, e.g. on mock backends
    print("Quality metrics not available:", e)

for key, value in info.items():
    print(
        f"{key:>28}: {value:.6g}" if isinstance(value, float) else f"{key:>28}: {value}"
    )

## Pulse shape

The I channel is a Hann window $\cos^2(\pi t)$ and the Q channel its normalized derivative
$-\sin(2\pi t)$, with $t \in [-1/2, 1/2]$ in units of the pulse duration. With
`amplitude_q = DRAG_RATIO * amplitude_i`, the Q channel adds a DRAG-like correction. Waveforms
return normalized shapes; the gate's `amplitude_i` / `amplitude_q` scale them.

In [ ]:
@dataclass(frozen=True)
class HannEnvelope(Waveform):
    # Hann window, 1 at the centre and 0 at both ends.

    def _sample(self, sample_coords: np.ndarray) -> np.ndarray:
        return np.cos(np.pi * sample_coords) ** 2


@dataclass(frozen=True)
class HannDerivative(Waveform):
    # Time derivative of HannEnvelope, normalized to [-1, 1].

    def _sample(self, sample_coords: np.ndarray) -> np.ndarray:
        return -np.sin(2 * np.pi * sample_coords)


class HannDragPRX(PRX_CustomWaveforms, wave_i=HannEnvelope, wave_q=HannDerivative):
    # PRX gate made of one Hann pulse (pi rotation at full amplitude), Q channel for DRAG.
    pass


n = 64
t = np.linspace(-0.5, 0.5, n)
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(t, HannEnvelope(n_samples=n).sample(), label="I: Hann")
ax.plot(t, HannDerivative(n_samples=n).sample(), label="Q: derivative")
ax.set_xlabel("time (pulse duration)")
ax.set_ylabel("normalized amplitude")
ax.legend()
plt.tight_layout()
plt.show()

## Register the gate and helpers

`add_implementation` makes `hann_drag` available to the compiler; the settings choose it as the
default `prx` implementation only when `custom=True`, so the native gate stays available for
comparison.

In [ ]:
compiler.add_implementation("prx", IMPL, HannDragPRX, overwrite=True)


def make_settings(circuits, *, shots, amplitude=None, custom=True):
    # Compiler settings using the custom gate (custom=True) or the native gate.
    s = compiler.get_settings(circuits=circuits)
    if custom:
        s.gate_definitions.prx.default_implementation = IMPL
        s.gates.prx[IMPL][QUBIT].set_from_dict(
            {
                "duration": DURATION,
                "amplitude_i": amplitude,
                "amplitude_q": DRAG_RATIO * amplitude,
            }
        )
    s.set_shots(shots)
    return s


def run(circuit, settings, sweeps=None):
    # Compile for QUBIT, submit through HEAppE and return the post-processed result.
    run_definition, context = compiler.compile(
        circuits=[circuit], components=[QUBIT], settings=settings, sweeps=sweeps or []
    )
    job = pulla.submit_playlist(run_definition, context=context)
    status = job.wait_for_completion(timeout_secs=JOB_TIMEOUT, polling_interval=2)
    if status is not JobStatus.COMPLETED:
        raise RuntimeError(f"Job {job.job_id} ended {status}: {job.job_data.messages}")
    return job.result_exa(compiler=compiler)


def excited_probability(result):
    # P(|1>) of QUBIT from the post-processed dataset (one value per sweep point).
    ds = result.dataset
    names = [
        name
        for name in ds.data_vars
        if name.startswith(f"{QUBIT}__") and name.endswith("excited_state_probability")
    ]
    if not names:
        raise KeyError(
            f"No excited state probability for {QUBIT}: {list(ds.data_vars)}"
        )
    return np.atleast_1d(np.asarray(ds[names[0]].values, dtype=float))


def peak_amplitude(amplitudes, probabilities):
    # Amplitude of the highest P(|1>): vertex of a parabola through the best point and its neighbours.
    i = int(np.argmax(probabilities))
    if 0 < i < len(amplitudes) - 1:
        a, b, _ = np.polyfit(amplitudes[i - 1 : i + 2], probabilities[i - 1 : i + 2], 2)
        if a < 0:
            return float(np.clip(-b / (2 * a), amplitudes[i - 1], amplitudes[i + 1]))
    return float(amplitudes[i])


qc_x = QuantumCircuit(1, 1, name="x")
qc_x.x(0)
qc_x.measure(0, 0)

qc_xx = QuantumCircuit(1, 1, name="xx")
qc_xx.x(0)
qc_xx.barrier()  # keeps the transpiler from cancelling the two X gates
qc_xx.x(0)
qc_xx.measure(0, 0)

## Rabi amplitude calibration

Each round sweeps `amplitude_i` (and `amplitude_q` with it) around the current best value and
takes the amplitude with the highest $P(1)$ after one X gate. Rounds get narrower and use more
shots. The native amplitude is only a starting point: the Hann pulse has a different area than
the native pulse shape, so the first round is wide.

In [ ]:
def rabi_round(center, span, n_points, shots):
    amplitudes = np.clip(np.linspace(center - span, center + span, n_points), 0.0, 1.0)
    s = make_settings([qc_x], shots=shots, amplitude=center)
    gate = s.gates.prx[IMPL][QUBIT]
    sweep = (  # swept together: one sweep point sets both channels
        Sweep(parameter=gate.amplitude_i.parameter, data=amplitudes),
        Sweep(parameter=gate.amplitude_q.parameter, data=DRAG_RATIO * amplitudes),
    )
    probabilities = excited_probability(run(qc_x, s, sweeps=[sweep]))
    best = peak_amplitude(amplitudes, probabilities)
    print(
        f"  center={center:.4f} span={span:.4f} -> best amplitude {best:.5f}, max P(1)={probabilities.max():.3f}"
    )
    return amplitudes, probabilities, best


start = NATIVE_AMPLITUDE if 0 < NATIVE_AMPLITUDE < 1 else 0.3
rounds = []
center = start
for span_factor, n_points, shots in (
    (0.6, 15, 500),
    (0.15, 11, 1000),
    (0.04, 11, 2000),
):
    print(f"Round {len(rounds) + 1}")
    amplitudes, probabilities, center = rabi_round(
        center, span_factor * start, n_points, shots
    )
    rounds.append((amplitudes, probabilities, center))

AMPLITUDE = center
print(
    f"\nCalibrated amplitude_i = {AMPLITUDE:.5f} (amplitude_q = {DRAG_RATIO * AMPLITUDE:.5f})"
)

In [ ]:
fig, axes = plt.subplots(1, len(rounds), figsize=(4 * len(rounds), 3.5))
for k, (ax, (amplitudes, probabilities, best)) in enumerate(zip(axes, rounds), start=1):
    ax.plot(amplitudes, probabilities, "o-")
    ax.axvline(best, color="tab:red", linestyle="--", label=f"A = {best:.4f}")
    ax.axhline(1.0, color="grey", linestyle=":")
    ax.set_title(f"Round {k}")
    ax.set_xlabel("amplitude_i")
    ax.set_ylabel("P(|1>) after X")
    ax.legend()
plt.tight_layout()
plt.show()

## Check against the native gate

A calibrated X gate gives $P(1) \approx 1$, and two of them give $P(1) \approx 0$ ($X^2 = I$).
The same circuits run with the native `prx` implementation for comparison.

In [ ]:
CHECK_SHOTS = 2000
checks = {}
for label, circuit in (("X", qc_x), ("X·X", qc_xx)):
    for gate_name, custom in ((IMPL, True), (NATIVE_IMPL, False)):
        s = make_settings(
            [circuit], shots=CHECK_SHOTS, amplitude=AMPLITUDE, custom=custom
        )
        checks[label, gate_name] = excited_probability(run(circuit, s))[0]

print(f"{'circuit':<8}{'expected':>10}{IMPL:>14}{NATIVE_IMPL:>18}")
for label, expected in (("X", 1.0), ("X·X", 0.0)):
    print(
        f"{label:<8}{expected:>10.1f}{checks[label, IMPL]:>14.4f}{checks[label, NATIVE_IMPL]:>18.4f}"
    )

## Save the calibration

Saves the gate parameters, so the gate can be registered again later with
`make_settings(..., amplitude=...)`.

In [ ]:
calibration = {
    "backend": pulla.quantum_computer_name,
    "qubit": QUBIT,
    "calibration_set_id": str(pulla.backend.architecture.calibration_set_id),
    "implementation": IMPL,
    "waveforms": {"i": "HannEnvelope", "q": "HannDerivative"},
    "duration_s": DURATION,
    "amplitude_i": AMPLITUDE,
    "amplitude_q": DRAG_RATIO * AMPLITUDE,
    "checks_p1": {f"{label} ({gate})": float(p) for (label, gate), p in checks.items()},
    "timestamp": datetime.now().isoformat(timespec="seconds"),
}
path = f"calibration_{IMPL}_{QUBIT}_{datetime.now():%Y%m%d_%H%M%S}.json"
with open(path, "w") as f:
    json.dump(calibration, f, indent=2)
print("Saved", path)
print(json.dumps(calibration, indent=2))